# Upgraded Solar Panel Analysis
This notebook provides:
1. **Baseline CNN Modeling**: Trains a model to classify panels as Clean or Dusty.
2. **Video Processing**: Processes a continuous video frame-by-frame.
3. **Panel Detection & Prediction**: Uses a sliding window (or contour detection) to find the panel, draws a red bounding box, and predicts its state visually inline.

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from IPython.display import display, Image, clear_output
import time
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

print("TensorFlow Version:", tf.__version__)
print("OpenCV Version:", cv2.__version__)

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("GPU is available and configured.")
else:
    print("No GPU found. Using CPU.")

## 1. Load Data & Build CNN Model
*(Re-using the proven architecture from our baseline case study)*

In [ ]:
DATASET_PATH = "Detect_solar_dust"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomTranslation(0.2, 0.2),
    tf.keras.layers.RandomZoom((-0.3, 0.3)),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH, validation_split=0.2, subset="training",
    seed=1337, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="binary"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH, validation_split=0.2, subset="validation",
    seed=1337, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="binary"
)

class_names = train_ds.class_names
print("Classes:", class_names)

def preprocess_func(images, labels, augment=False):
    if augment:
        images = data_augmentation(images, training=True)
    return preprocess_input(tf.cast(images, tf.float32)), labels

AUTOTUNE = tf.data.AUTOTUNE
train_data = train_ds.map(lambda x, y: preprocess_func(x, y, True), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_data = val_ds.map(lambda x, y: preprocess_func(x, y, False), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

In [ ]:
def build_model():
    base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False
    
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation="sigmoid", dtype="float32")(x)
    
    model = Model(inputs=base_model.input, outputs=output)
    model.compile(optimizer=Adam(5e-5), loss="binary_crossentropy", metrics=["accuracy"])
    return model

model = build_model()

# If we already have the previously trained weights, we can load them to save time.
WEIGHTS_PATH = "best_resnet_solar.keras"
if os.path.exists(WEIGHTS_PATH):
    print(f"Loading existing weights from {WEIGHTS_PATH}...")
    model.load_weights(WEIGHTS_PATH)
else:
    print("No pre-trained weights found. You will need to train the model using `model.fit(train_data, validation_data=val_data, epochs=5)`")


## 2. Video Processing & Sliding Window Pipeline
We use OpenCV to isolate the solar panel. With the updated code below, we also display the frames **inline** in Jupyter so you can visually see the model locating the panel and classifying it in real-time.

In [ ]:
def predict_on_roi(roi_img, model, preprocess_fn, img_size):
    """Prepares a cropped BGR ROI image for the ResNet50 model and returns the prediction."""
    rgb_roi = cv2.cvtColor(roi_img, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb_roi, img_size)
    input_tensor = preprocess_fn(tf.cast(np.expand_dims(resized, axis=0), tf.float32))
    pred = model.predict(input_tensor, verbose=0)[0][0]
    predicted_class = "Dusty" if pred > 0.5 else "Clean"
    confidence = pred if pred > 0.5 else 1.0 - pred
    return predicted_class, confidence

def locate_solar_panel(frame):
    """
    Simple baseline object localization: 
    Uses edge detection and contours to find the largest rectangular-ish object assuming it's the solar panel.
    Returns (x, y, w, h) of the bounding box, or None if not found.
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blur, 50, 150)
    
    kernel = np.ones((5,5), np.uint8)
    dilated = cv2.dilate(edges, kernel, iterations=1)
    
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
        
    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    largest_contour = contours[0]
    
    if cv2.contourArea(largest_contour) < 5000:
        return None
        
    x, y, w, h = cv2.boundingRect(largest_contour)
    return (x, y, w, h)


In [ ]:
def process_video_visually(input_video_path, output_video_path, model, preprocess_fn, img_size):
    """
    Reads a video frame by frame, detects the solar panel, classifies it, 
    draws a red box with the prediction, outputs the stream visually to Jupyter, 
    and saves the output.
    """
    if not os.path.exists(input_video_path):
        print(f"Error: Video {input_video_path} not found.")
        return
        
    cap = cv2.VideoCapture(input_video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    print(f"Processing {total_frames} frames...")
    
    # Setup the inline display handler
    display_handle = display(None, display_id=True)
    frame_count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        # 1. Locate Panel
        bbox = locate_solar_panel(frame)
        
        if bbox:
            x, y, w, h = bbox
            roi = frame[y:y+h, x:x+w]
            
            # 2. Predict on the isolated panel
            if w > 50 and h > 50:
                label, confidence = predict_on_roi(roi, model, preprocess_fn, img_size)
                
                color = (0, 0, 255) # Red for Dusty, Green for Clean could be an upgrade
                if label == "Clean":
                    color = (0, 255, 0)

                # 3. Draw Bounding Box
                cv2.rectangle(frame, (x, y), (x+w, y+h), color, 3)
                
                # 4. Draw Label Text
                text = f"{label} ({confidence:.2f})"
                (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)
                cv2.rectangle(frame, (x, y - th - 10), (x + tw, y), color, -1)
                cv2.putText(frame, text, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        
        # Write processed frame to output video file
        out.write(frame)
        
        # Convert frame for Jupyter display
        # Resize heavily if it's too big so it runs smoothly in browser
        display_frame = cv2.resize(frame, (640, int(640 * (height/width))))
        _, encoded_image = cv2.imencode('.jpg', display_frame)
        display_handle.update(Image(data=encoded_image.tobytes()))
        
        # Small artificial delay to slow it down so user can actually watch it
        time.sleep(0.02)
        
        frame_count += 1

    cap.release()
    out.release()
    print(f"\nVideo processing complete. Saved to {output_video_path}")


In [ ]:
# TEST EXECUTION
input_video = "Solar Panel Videos/Clean Solar Panel 1.mp4" 
output_video = "Solar Panel Videos/processed_output.mp4"

print(f"Running visual pipeline on {input_video}...")
process_video_visually(input_video, output_video, model, preprocess_input, IMG_SIZE) # Passing the expected specific dependencies
